In [795]:
# TODO:
# 1. Add DWI unit model to the flowsheet

In [796]:
# IMPORTS FROM PYOMO

from pyomo.environ import (
    ConcreteModel,
    Var,
    Param,
    Constraint,
    Objective,
    Expression,
    value,
    check_optimal_termination,
    assert_optimal_termination,
    TransformationFactory,
    units as pyunits,
)

from pyomo.util.check_units import assert_units_consistent
from pyomo.network import Arc


# IMPORTS FROM IDAES
from idaes.core import FlowsheetBlock, UnitModelCostingBlock
from idaes.models.unit_models import Feed, Product, StateJunction
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.core.util.scaling import calculate_scaling_factors, set_scaling_factor, constraint_scaling_transform
from idaes.core.util.initialization import propagate_state


# IMPORTS FROM WaterTAP
from watertap.property_models.NaCl_prop_pack import NaClParameterBlock
from watertap.property_models.seawater_prop_pack import SeawaterParameterBlock
from watertap.unit_models.pressure_changer import Pump
from watertap.unit_models.reverse_osmosis_0D import (
    ReverseOsmosis0D,
    ConcentrationPolarizationType,
    MassTransferCoefficient,
    PressureChangeType,
)

from watertap.unit_models.zero_order import ChemicalAdditionZO
from watertap.unit_models.zero_order import UltraFiltrationZO
from watertap.unit_models.zero_order import ChlorinationZO
from watertap.unit_models.zero_order import DeepWellInjectionZO

# What is this line for? 
# MH: This is for loading the ZO models database. ZO models have default parameters that are stored in a database. 
# MH: If you follow this path in the watertap repository, you can see the database file: src/watertap/core/wt_database.py
from watertap.core.wt_database import Database                                                              

from watertap.core.zero_order_properties import WaterParameterBlock as ZOProperties
from watertap.costing.zero_order_costing import ZeroOrderCosting
from watertap.costing import WaterTAPCosting
from watertap.core.solvers import get_solver


# TRANSLATOR FUNCTION
import idaes.logger as idaeslog
from idaes.core import declare_process_block_class
from idaes.core.util.exceptions import InitializationError
from idaes.models.unit_models.translator import TranslatorData

from watertap.core.util.model_diagnostics.infeasible import *
from idaes.core.util.model_diagnostics import DiagnosticsToolbox

In [797]:
# Translator function: Z0 to NaCl unit property block
# MH: Updated this cell to be for NaCl Property Block instead of Seawater property block. 
# MH: This translator will be used to translate from the ZO property block to the NaCl property block, which will then be used in the RO unit model.

@declare_process_block_class("TranslatorZOtoNaCl")
class TranslatorZOtoNaClData(TranslatorData):
    """
    Translator block for converting from ZO TDS to NaCl
    """

    CONFIG = TranslatorData.CONFIG()

    def build(self):
        super().build()

        @self.Constraint(
            self.flowsheet().time,
            doc="Equality mass flow water equation",
        )
        def eq_flow_mass_rule(blk, t):
            return (
                blk.properties_out[t].flow_mass_phase_comp["Liq", "H2O"]
                == blk.properties_in[t].flow_mass_comp["H2O"]
            )

        @self.Constraint(
            self.flowsheet().time,
            doc="Equality solute equation",
        )
        def eq_solute_mass_flow(blk, t):
            return (
                blk.properties_out[t].flow_mass_phase_comp["Liq", "NaCl"]
                == blk.properties_in[t].flow_mass_comp["NaCl"]
            )

        # ZO prop pack doesn't have temperature and pressure as state variables
        self.properties_out[0].pressure.fix(101325)
        self.properties_out[0].temperature.fix(298.15)

    def initialize_build(
        self,
        state_args_in=None,
        state_args_out=None,
        outlvl=idaeslog.NOTSET,
        solver=None,
        optarg=None,
    ):
        init_log = idaeslog.getInitLogger(self.name, outlvl, tag="unit")

        # Create solver
        opt = get_solver(solver, optarg)

        # ---------------------------------------------------------------------
        # Initialize state block
        flags = self.properties_in.initialize(
            outlvl=outlvl,
            optarg=optarg,
            solver=solver,
            state_args=state_args_in,
            hold_state=True,
        )

        self.properties_out.initialize(
            outlvl=outlvl,
            optarg=optarg,
            solver=solver,
            state_args=state_args_out,
        )

        if degrees_of_freedom(self) != 0:
            raise Exception(
                f"{self.name} degrees of freedom were not 0 at the beginning "
                f"of initialization. DoF = {degrees_of_freedom(self)}"
            )

        with idaeslog.solver_log(init_log, idaeslog.DEBUG) as slc:
            res = opt.solve(self, tee=slc.tee)

        self.properties_in.release_state(flags=flags, outlvl=outlvl)

        init_log.info(f"Initialization Complete: {idaeslog.condition(res)}")

        if not check_optimal_termination(res):
            raise InitializationError(
                f"{self.name} failed to initialize successfully. Please check "
                f"the output logs for more information."
            )

In [798]:
## MH: I basically flipped the outlet and inlet properties from the TranslatorZOtoSW to create the TranslatorZOtoNaCl.
# Translator function: NaCl to ZO unit property block


@declare_process_block_class("TranslatorNaCltoZO")
class TranslatorNaCltoZOData(TranslatorData):
    """
    Translator block for converting from NaCl to ZO TDS
    """

    CONFIG = TranslatorData.CONFIG()

    def build(self):
        super().build()

        @self.Constraint(
            self.flowsheet().time,
            doc="Equality mass flow water equation",
        )
        def eq_flow_mass_rule(blk, t):
            return (
                blk.properties_in[t].flow_mass_phase_comp["Liq", "H2O"]
                == blk.properties_out[t].flow_mass_comp["H2O"]
            )

        @self.Constraint(
            self.flowsheet().time,
            doc="Equality solute equation",
        )
        def eq_solute_mass_flow(blk, t):
            return (
                blk.properties_in[t].flow_mass_phase_comp["Liq", "NaCl"]
                == blk.properties_out[t].flow_mass_comp["NaCl"]
            )

        # ZO prop pack doesn't have temperature and pressure as state variables
        # self.properties_out[0].pressure.fix(101325)
        # self.properties_out[0].temperature.fix(298.15)

    def initialize_build(
        self,
        state_args_in=None,
        state_args_out=None,
        outlvl=idaeslog.NOTSET,
        solver=None,
        optarg=None,
    ):
        init_log = idaeslog.getInitLogger(self.name, outlvl, tag="unit")

        # Create solver
        opt = get_solver(solver, optarg)

        # ---------------------------------------------------------------------
        # Initialize state block
        flags = self.properties_in.initialize(
            outlvl=outlvl,
            optarg=optarg,
            solver=solver,
            state_args=state_args_in,
            hold_state=True,
        )

        self.properties_out.initialize(
            outlvl=outlvl,
            optarg=optarg,
            solver=solver,
            state_args=state_args_out,
        )

        if degrees_of_freedom(self) != 0:
            raise Exception(
                f"{self.name} degrees of freedom were not 0 at the beginning "
                f"of initialization. DoF = {degrees_of_freedom(self)}"
            )

        with idaeslog.solver_log(init_log, idaeslog.DEBUG) as slc:
            res = opt.solve(self, tee=slc.tee)

        self.properties_in.release_state(flags=flags, outlvl=outlvl)

        init_log.info(f"Initialization Complete: {idaeslog.condition(res)}")

        if not check_optimal_termination(res):
            raise InitializationError(
                f"{self.name} failed to initialize successfully. Please check "
                f"the output logs for more information."
            )

In [799]:
# MODEL PARAMETERS

solute_list = ["NaCl"]

mass_flow_water = 0.965 * pyunits.kg / pyunits.s
mass_flow_salt = 0.035 * pyunits.kg / pyunits.s

use_default_removal = True

# Chemical dosing parameters
anti_scalant_dose = 1 * pyunits.mg / pyunits.liter

# Pump parameters
pump_efficiency = 0.85 * pyunits.dimensionless
operating_pressure = 75 * pyunits.bar

# RO parameters

A_comp =4.2e-12 * pyunits.m/(pyunits.s * pyunits.Pa)                    # membrane water permeability coefficient [m/s-Pa]
B_comp = 3e-8 * pyunits.m/(pyunits.s)                                   # membrane salt permeability coefficient [m/s]
membrane_area = 50  * pyunits.m**2
atmospheric = 101325  * pyunits.Pa
deltaP = -3 * pyunits.bar
channel_height = 1 * pyunits.mm 
spacer_porosity = 0.75  * pyunits.dimensionless
RR = 0.45   * pyunits.dimensionless

solver = get_solver()


In [800]:
m = ConcreteModel()
# What is this line for?
# This creates a database object that can be used to access the parameters for the ZO models. 
# The ZO models have default parameters that are stored in a yaml file, and this line allows us to access those parameters when we create the ZO unit models.
m.db = Database()                                                                                      
m.fs = FlowsheetBlock(dynamic=False)

# Add property models
m.fs.zo_properties = ZOProperties(solute_list=solute_list)
m.fs.ro_properties = NaClParameterBlock()

# Add unit models
m.fs.feed = Feed(property_package=m.fs.zo_properties)
# Set feed stream
m.fs.feed.properties[0].flow_mass_comp["H2O"].fix(mass_flow_water)
m.fs.feed.properties[0].flow_mass_comp["NaCl"].fix(mass_flow_salt)

# Where to see arguments required for each function
# MH: You can look at the ZO model in the watertap repository to see what arguments are required for the constructor. 
# MH: For the UltraFiltrationZO model, you can find it here: https://github.com/watertap-org/watertap/blob/main/watertap/data/techno_economic/ultra_filtration.yaml
m.fs.ultra_filt = UltraFiltrationZO(property_package=m.fs.zo_properties, database=m.db)    

# Where to see arguments required for each function
# MH: You can look at the ZO model in the watertap repository to see what arguments are required for each function
# MH: For the ChemicalAdditionZO model, you can find it here: https://github.com/watertap-org/watertap/blob/main/watertap/data/techno_economic/chemical_addition.yaml
m.fs.chem_addition = ChemicalAdditionZO(property_package=m.fs.zo_properties,database=m.db, process_subtype="anti-scalant")           
m.fs.chem_addition.chemical_dosage.fix(anti_scalant_dose)

# How to fix this translation to NaCl instead of SW
# MH: You had already copied over the code correctly for the translator, so I just updated the class name to TranslatorZOtoNaCl and made sure the property packages were correct.
m.fs.translator_feed = TranslatorZOtoNaCl(inlet_property_package=m.fs.zo_properties,outlet_property_package=m.fs.ro_properties)       


m.fs.pump = Pump(property_package=m.fs.ro_properties)
m.fs.pump.efficiency_pump.fix(pump_efficiency)

# Hydrostatic pressure at the pump inlet
# m.fs.pump.control_volume.properties_in[0].pressure.fix(101325) 
m.fs.pump.control_volume.properties_out[0].pressure.fix(operating_pressure)

m.fs.RO = ReverseOsmosis0D(
    property_package=m.fs.ro_properties,
    has_pressure_change=True,
    pressure_change_type=PressureChangeType.calculated,
    mass_transfer_coefficient=MassTransferCoefficient.calculated,
    concentration_polarization_type=ConcentrationPolarizationType.calculated,
    module_type="spiral_wound",
)

# Fix (2) membrane properties
m.fs.RO.A_comp.fix(A_comp)
m.fs.RO.B_comp.fix(B_comp)

# Fix (4) module specifications
m.fs.RO.feed_side.channel_height.fix(channel_height)
m.fs.RO.feed_side.spacer_porosity.fix(spacer_porosity)
m.fs.RO.area.fix(membrane_area)
m.fs.RO.deltaP.fix(deltaP)                           
# m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(RR)

# (1) outlet state variable
m.fs.RO.permeate.pressure[0].fix(atmospheric)


# Create an Expression to calculate flux in LMH
m.fs.RO.flux_LMH = Expression(
    expr=pyunits.convert(
        m.fs.RO.mixed_permeate[0].flow_vol_phase["Liq"] / m.fs.RO.area,
        to_units=pyunits.liter / (pyunits.m**2 * pyunits.hr),
    )
)

## MH: Updated product to be a state junction
# Feed - Only an outlet
# Product - Only an inlet
# State Junction - both inlet and outlet, can be used to connect unit models without changing the state variables.
m.fs.product = StateJunction(property_package=m.fs.ro_properties)
m.fs.product.properties[0].conc_mass_phase_comp     

# m.fs.brine = StateJunction(property_package=m.fs.ro_properties)
# m.fs.dwi = DeepWellInjectionZO(property_package=m.fs.zo_properties, database=m.db)

# Connect unit models with Arcs
m.fs.feed_to_ultra = Arc(source=m.fs.feed.outlet, destination=m.fs.ultra_filt.inlet)
m.fs.ultra_to_chem = Arc(source=m.fs.ultra_filt.treated, destination=m.fs.chem_addition.inlet)
m.fs.chem_to_trans = Arc(source=m.fs.chem_addition.outlet, destination=m.fs.translator_feed.inlet)
m.fs.trans_to_pump = Arc(source=m.fs.translator_feed.outlet, destination=m.fs.pump.inlet)
m.fs.pump_to_RO = Arc(source=m.fs.pump.outlet, destination=m.fs.RO.inlet)
m.fs.RO_to_product = Arc(source=m.fs.RO.permeate, destination=m.fs.product.inlet)                      


# How to add chlorination and remineralization after this?
# MH: You would need to add the chlorination and remineralization unit models after the RO unit model, and then connect them with Arcs.
m.fs.chlorination = ChemicalAdditionZO(property_package=m.fs.zo_properties, database=m.db, process_subtype="alum") 
m.fs.chlorination.chemical_dosage.fix(1 * pyunits.mg / pyunits.liter)   

# MH: Create a translator from the RO property package to the ZO property package to connect the RO unit model to the chlorination
m.fs.translator_product = TranslatorNaCltoZO(inlet_property_package=m.fs.ro_properties,outlet_property_package=m.fs.zo_properties) 
# m.fs.translator_brine = TranslatorNaCltoZO(inlet_property_package=m.fs.ro_properties,outlet_property_package=m.fs.zo_properties)


# MH: Connect the product to the translator, and then connect the translator to the chlorination
m.fs.product_to_translator = Arc(source=m.fs.RO.permeate, destination=m.fs.translator_product.inlet)
m.fs.translator_to_chlorination = Arc(source=m.fs.translator_product.outlet, destination=m.fs.chlorination.inlet)


# # Brine ARCs
# m.fs.RO_to_brine = Arc(source=m.fs.RO.retentate, destination=m.fs.translator_brine.inlet)
# m.fs.translator_brine_to_dwi = Arc(source=m.fs.translator_brine.outlet, destination=m.fs.dwi.inlet)

TransformationFactory("network.expand_arcs").apply_to(m)       

In [801]:
# Set scaling factors
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1, index=("Liq", "H2O"))
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e2, index=("Liq", "NaCl"))

m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1, index=("H2O"))
m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1e2, index=("NaCl"))

set_scaling_factor(m.fs.pump.control_volume.work, 1e-3)
set_scaling_factor(m.fs.RO.area, 1e-2)

calculate_scaling_factors(m)

# Release constraints related to low concentrations
for item in [m.fs.RO.permeate_side, m.fs.RO.feed_side.properties_interface]:
    for idx, param in item.items():
        param.molality_phase_comp["Liq", "NaCl"].setlb(1e-5)   ## None maybe...
        param.pressure_osm_phase["Liq"].setlb(10)

print(f"dof = {degrees_of_freedom(m)}")

dof = 7


In [802]:
solver = get_solver()
   
################Added lines to initialize unit models in sequence##################
m.fs.feed.properties[0].flow_vol
m.fs.feed.initialize()
####################################################################################
propagate_state(m.fs.feed_to_ultra)
# What do I need this line?
# MH: You need to load parameters from the database for the ZO models before initializing them, 
# otherwise they will be initialized with default parameters which may not be correct for your case.
m.fs.ultra_filt.load_parameters_from_database(use_default_removal=use_default_removal)     
m.fs.ultra_filt.initialize()

propagate_state(m.fs.ultra_to_chem)
m.fs.chem_addition.load_parameters_from_database(use_default_removal=use_default_removal)

# MH: We need to update the chemical dosage for the chemical addition unit model before initializing it,
# otherwise it will be initialized with the default chemical dosage which may not be correct for your case.       
m.fs.chem_addition.chemical_dosage.fix(anti_scalant_dose) 
m.fs.chem_addition.initialize()

propagate_state(m.fs.chem_to_trans)
m.fs.translator_feed.initialize()

propagate_state(m.fs.trans_to_pump)
m.fs.pump.initialize()

propagate_state(m.fs.pump_to_RO)

print(f"dof = {degrees_of_freedom(m)}")
m.fs.RO.initialize()
print(f"dof = {degrees_of_freedom(m.fs.RO)}")

propagate_state(m.fs.RO_to_product)
m.fs.product.initialize()

propagate_state(m.fs.product_to_translator)
m.fs.translator_product.initialize()

propagate_state(m.fs.translator_to_chlorination)
# ## MH: Following the same steps as the chemical additiona block above
m.fs.chlorination.load_parameters_from_database(use_default_removal=use_default_removal)
m.fs.chlorination.chemical_dosage.fix(1 * pyunits.mg / pyunits.liter) 
m.fs.chlorination.initialize()

## Initialize DWI components

2026-03-04 12:19:09 [INFO] idaes.init.fs.feed.properties: Initialization Complete.
2026-03-04 12:19:09 [INFO] idaes.init.fs.feed: Initialization Complete.
2026-03-04 12:19:09 [INFO] idaes.init.fs.ultra_filt.properties_in: Initialization Complete.
2026-03-04 12:19:09 [INFO] idaes.init.fs.ultra_filt.properties_treated: Initialization Complete.
2026-03-04 12:19:09 [INFO] idaes.init.fs.ultra_filt.properties_byproduct: Initialization Complete.
2026-03-04 12:19:09 [INFO] idaes.init.fs.ultra_filt.properties_in: State Released.
2026-03-04 12:19:09 [INFO] idaes.init.fs.ultra_filt: Initialization Complete: optimal - Optimal Solution Found
2026-03-04 12:19:09 [INFO] idaes.init.fs.chem_addition.properties: Initialization Complete.
2026-03-04 12:19:09 [INFO] idaes.init.fs.chem_addition.properties: State Released.
2026-03-04 12:19:09 [INFO] idaes.init.fs.chem_addition: Initialization Complete: optimal - Optimal Solution Found
2026-03-04 12:19:09 [INFO] idaes.init.fs.translator_feed.properties_in: In

In [803]:
print(f"dof = {degrees_of_freedom(m)}")

dof = 0


In [804]:
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

In [805]:
# Add costing
# m.fs.costing = ZeroOrderCosting()
# m.fs.costing.base_currency = pyunits.USD_2023

# m.fs.ultra_filt.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)
# # m.fs.chem_addition.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)
# m.fs.pump.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)
# m.fs.RO.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)

# m.fs.chlorination.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)
# # m.fs.dwi.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)

# m.fs.costing.cost_process()
# m.fs.costing.add_LCOW(m.fs.product.properties[0].flow_vol_phase["Liq"])
# m.fs.costing.add_specific_energy_consumption(m.fs.product.properties[0].flow_vol_phase["Liq"], name="SEC")

In [806]:
m.fs.pump.control_volume.properties_out[0].pressure.unfix()
m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(RR)
print(f"dof = {degrees_of_freedom(m)}")

dof = 0


In [807]:
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

In [808]:
m.fs.chem_addition.chemical_flow_vol.display()

chemical_flow_vol : Volumetric flow rate of chemical solution
    Size=1, Index=fs._time, Units=m**3/s
    Key : Lower : Value                 : Upper : Fixed : Stale : Domain
    0.0 :     0 : 9.321751388833728e-10 :  None : False : False :  Reals


In [809]:
# Update feed concentration and re-solve
m.fs.feed.properties[0].flow_mass_comp["NaCl"].fix(mass_flow_salt/1000)

m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1, index=("Liq", "H2O"))
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e5, index=("Liq", "NaCl"))
m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1, index=("H2O"))
m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1e5, index=("NaCl"))

calculate_scaling_factors(m)
m.fs.feed.initialize()

2026-03-04 12:19:10 [INFO] idaes.init.fs.feed.properties: Initialization Complete.
2026-03-04 12:19:10 [INFO] idaes.init.fs.feed: Initialization Complete.


In [810]:
m.fs.chem_addition.chemical_flow_vol.display()

chemical_flow_vol : Volumetric flow rate of chemical solution
    Size=1, Index=fs._time, Units=m**3/s
    Key : Lower : Value                 : Upper : Fixed : Stale : Domain
    0.0 :     0 : 9.321751388833728e-10 :  None : False : False :  Reals


In [811]:
# MH: Update scaling factors for the chemical addition unit model
set_scaling_factor(m.fs.chem_addition.chemical_flow_vol, 1e10)
set_scaling_factor(m.fs.chlorination.chemical_flow_vol, 1e10)
print(f"dof = {degrees_of_freedom(m)}")

dof = 0


In [812]:
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

In [813]:
print_infeasible_constraints(m)

## High flowrate
Inlet feed flow rate of 500 m3/h
Recovery ratio of 90 or 95%

In [814]:
flow_vol = 500*pyunits.m**3/pyunits.hour
density = 1000*pyunits.kg/pyunits.m**3

feed_conc = 5 * pyunits.mg/pyunits.liter

mass_flow_water = pyunits.convert(flow_vol * density, to_units=pyunits.kg/pyunits.s) 
mass_flow_salt = pyunits.convert(feed_conc, to_units=pyunits.kg/pyunits.m**3) * pyunits.convert(flow_vol, to_units=pyunits.m**3/pyunits.s)
print("Mass flow of water: ", value(mass_flow_water), pyunits.get_units(mass_flow_water))
print("Mass flow of salt: ", value(mass_flow_salt), pyunits.get_units(mass_flow_salt))

RR = 0.9

Mass flow of water:  138.88888888888889 kg/s
Mass flow of salt:  0.0006944444444444444 kg/s


In [816]:
m.fs.feed.properties[0].flow_mass_comp["H2O"].fix(mass_flow_water)
m.fs.feed.properties[0].flow_mass_comp["NaCl"].fix(mass_flow_salt)
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e-3, index=("Liq", "H2O"))
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e5, index=("Liq", "NaCl"))
m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1e-3, index=("H2O"))
m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1e5, index=("NaCl"))

# set_scaling_factor(m.fs.chem_addition.chemical_flow_vol, 1e11)
# set_scaling_factor(m.fs.chlorination.chemical_flow_vol, 1e11)

# Release constraints related to low concentrations
for item in [m.fs.RO.permeate_side, m.fs.RO.feed_side.properties_interface]:
    for idx, param in item.items():
        param.molality_phase_comp["Liq", "NaCl"].setlb(1e-6)   ## None maybe...
        param.pressure_osm_phase["Liq"].setlb(None)


m.fs.RO.feed_side.dh.setlb(None)

# m.fs.pump.control_volume.properties_out[0].pressure.unfix()
# m.fs.RO.recovery_vol_phase[0.0, "Liq"].unfix()

# m.fs.RO.area.unfix()

calculate_scaling_factors(m)
m.fs.feed.initialize()
propagate_state(m.fs.feed_to_ultra)
# m.fs.ultra_filt.load_parameters_from_database(use_default_removal=use_default_removal)     
# m.fs.ultra_filt.initialize()

2026-03-04 12:19:38 [INFO] idaes.init.fs.feed.properties: Initialization Complete.
2026-03-04 12:19:38 [INFO] idaes.init.fs.feed: Initialization Complete.


In [817]:
solver = get_solver()
results = solver.solve(m)
# assert_optimal_termination(results)

model.name="unknown";
    - termination condition: infeasible
    - message from solver: Ipopt 3.13.2\x3a Converged to a locally infeasible
      point. Problem may be infeasible.


In [818]:
print_infeasible_constraints(m)

CONSTR fs.RO.eq_recovery_vol_phase[0.0]: 0.44000001022682644 =/= 0.0
CONSTR fs.RO.eq_mass_frac_permeate[0.0,0.0,NaCl]: 0.00012167288609353464 =/= 0.0
CONSTR fs.RO.eq_mass_frac_permeate[0.0,1.0,NaCl]: 0.0001179993538923421 =/= 0.0
CONSTR fs.RO.feed_side.eq_dh: 0.002272850679888681 =/= 0.0
CONSTR fs.RO.feed_side.eq_area: 0.0690163699921783 =/= 0.0
CONSTR fs.RO.feed_side.properties_out[0.0].eq_pressure_osm_phase[Liq]: 4.486325313989772e-05 =/= 0.0
CONSTR fs.RO.feed_side.properties_out[0.0].eq_molality_phase_comp[Liq,NaCl]: 5.378244881084473e-05 =/= 0.0
